# Demo 3: Multi-Agent Comparison (เปรียบเทียบหลาย Agent)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiraphap/Gen-Agent-Lectures-at-Econ-TU/blob/master/teaching/demos/demo_multi_agent.ipynb)

---

## Overview

สคริปต์นี้สาธิตว่า **Agent ที่มี personality ต่างกัน จะตัดสินใจต่างกันอย่างไร** แม้จะเจอสถานการณ์เดียวกัน

### 3 Agent ที่ใช้:
1. **สมศรี** (แม่บ้าน) - ดูราคาเป็นหลัก, ชอบโปร
2. **สมชาย** (โปรแกรมเมอร์) - เน้นความสะดวก, ชอบของดี
3. **ลุงสมบัติ** (ข้าราชการเกษียณ) - ประหยัดสุดๆ, ภักดีต่อแบรนด์

### สถานการณ์เดียวกัน:
> "ต้องซื้อนม วันเสาร์เช้า"

แต่แต่ละคนตัดสินใจไม่เหมือนกัน!

### Concept หลัก:
- **Personality Traits** (0.0-1.0) กำหนดแนวโน้มพฤติกรรม
- **LLM** พิจารณาทุกปัจจัยพร้อมกัน ไม่ใช่ if-else
- **Memory + Situation** ส่งผลต่อการตัดสินใจ

---
**ผศ.ดร.ถิรภาพ ฟักทอง** | คณะเศรษฐศาสตร์ มหาวิทยาลัยธรรมศาสตร์

In [ ]:
# ============================================================
# Config + Imports + Helper Functions
# ============================================================

import json
import os
from typing import Dict, List, Optional
from dataclasses import dataclass

# ============================================================
# TOGGLE: เปิด/ปิดการใช้ LLM จริง
# ============================================================
# False = ใช้ข้อมูล mock (ไม่ต้องมี API key)
# True  = เรียก OpenAI API จริง (ต้องมี OPENAI_API_KEY)
USE_REAL_LLM = False


# ============================================================
# Helper Functions (ไม่มี ANSI colors - ใช้ Unicode bar charts)
# ============================================================

def print_header(text: str):
    """แสดงหัวข้อหลัก"""
    width = 64
    print(f"\n{'=' * width}")
    print(f"  {text}")
    print(f"{'=' * width}")


def print_subheader(text: str):
    """แสดงหัวข้อย่อย"""
    print(f"\n  -- {text} --")


def make_bar(value: float, max_width: int = 20) -> str:
    """สร้าง bar chart ด้วย Unicode"""
    filled = int(value * max_width)
    bar = "█" * filled + "░" * (max_width - filled)
    return bar


def display_personality_radar(agent: Dict):
    """แสดง personality traits แบบ text-based bar chart

    ใช้ bar chart แสดง trait แต่ละตัว
    ทำให้เห็นภาพรวมบุคลิกของ agent ได้ง่าย
    """
    name = agent["name"]
    p = agent["personality"]

    print(f"\n  >>> {name} ({agent['job']}, อายุ {agent['age']}, "
          f"รายได้ {agent['monthly_income']:,} บ./ด.)")
    print(f"      \"{agent['background']}\"\n")

    # แสดง trait labels ภาษาไทย
    trait_labels = {
        "price_sensitivity": "ดูราคา (Price Sensitivity)    ",
        "brand_loyalty": "ภักดีแบรนด์ (Brand Loyalty)   ",
        "quality_preference": "ชอบคุณภาพ (Quality Pref)     ",
        "convenience_preference": "เน้นสะดวก (Convenience Pref) ",
        "promotion_attraction": "ชอบโปร (Promo Attraction)    ",
        "impulse_buying": "ซื้อตามอารมณ์ (Impulse Buy)  ",
    }

    for trait_key, label in trait_labels.items():
        value = p.get(trait_key, 0)
        bar = make_bar(value, 20)
        print(f"    {label} {bar} {value:.2f}")

    # แสดง innate traits
    print(f"    {'─' * 50}")
    print(f"    นิสัย: {', '.join(agent.get('innate_traits', []))}")
    print(f"    ไม่ชอบ: {', '.join(agent.get('pain_points', []))}")


def display_thinking_process(agent_name: str, decision_data: Dict):
    """แสดงกระบวนการคิดของ agent"""
    print(f"\n  >>> {agent_name} กำลังคิด...\n")

    for i, step in enumerate(decision_data.get("thinking_steps", []), 1):
        print(f"    {i}. {step}")

    d = decision_data.get("decision", {})
    print(f"\n    ┌{'─' * 52}┐")
    print(f"    │ การตัดสินใจ:                                       │")
    print(f"    │   ไป: {d.get('store', 'N/A'):<46s}│")
    items_str = ", ".join(d.get("items_to_buy", []))
    if len(items_str) > 44:
        items_str = items_str[:41] + "..."
    print(f"    │   ซื้อ: {items_str:<45s}│")
    spend_str = f"~{d.get('estimated_spend', 0)} บาท"
    print(f"    │   งบ: {spend_str:<46s}│")
    print(f"    └{'─' * 52}┘")


print("Config loaded. USE_REAL_LLM =", USE_REAL_LLM)

In [ ]:
# ============================================================
# ข้อมูล Agent 3 คน
# ============================================================
# แต่ละคนมี personality traits ที่ต่างกัน
# ส่งผลต่อการตัดสินใจผ่าน LLM

AGENTS = {
    "สมศรี": {
        "name": "สมศรี",
        "age": 45,
        "job": "แม่บ้าน",
        "household_size": 4,
        "monthly_income": 35000,
        "personality": {
            "price_sensitivity": 0.8,
            "brand_loyalty": 0.6,
            "quality_preference": 0.5,
            "convenience_preference": 0.4,
            "promotion_attraction": 0.9,
            "impulse_buying": 0.3
        },
        "innate_traits": ["ใจเย็น", "ละเอียดรอบคอบ", "ชอบเปรียบเทียบราคา"],
        "background": "แม่บ้านดูแลครอบครัว 4 คน ดูใบปลิวโปรทุกสัปดาห์ ชอบเปรียบเทียบราคาก่อนซื้อ",
        "pain_points": ["ไม่ชอบร้านที่จอดรถยาก", "หงุดหงิดถ้าของหมด"],
        "preferred_stores": ["CJ More", "Big C"],
    },
    "สมชาย": {
        "name": "สมชาย",
        "age": 32,
        "job": "โปรแกรมเมอร์",
        "household_size": 1,
        "monthly_income": 80000,
        "personality": {
            "price_sensitivity": 0.3,
            "brand_loyalty": 0.4,
            "quality_preference": 0.8,
            "convenience_preference": 0.9,
            "promotion_attraction": 0.2,
            "impulse_buying": 0.6
        },
        "innate_traits": ["ใจร้อน", "ไม่ชอบเสียเวลา", "ชอบเทคโนโลยี"],
        "background": "โปรแกรมเมอร์อยู่คอนโดคนเดียว ยอมจ่ายแพงกว่าเพื่อความสะดวก ไม่ค่อยมีเวลา",
        "pain_points": ["เกลียดต่อคิวยาว", "ไม่ชอบร้านไกล", "ไม่มีเวลาเดินเลือกนาน"],
        "preferred_stores": ["7-Eleven", "Tops"],
    },
    "ลุงสมบัติ": {
        "name": "ลุงสมบัติ",
        "age": 62,
        "job": "ข้าราชการเกษียณ",
        "household_size": 2,
        "monthly_income": 15000,
        "personality": {
            "price_sensitivity": 0.95,
            "brand_loyalty": 0.8,
            "quality_preference": 0.4,
            "convenience_preference": 0.3,
            "promotion_attraction": 0.6,
            "impulse_buying": 0.1
        },
        "innate_traits": ["ประหยัด", "มีระเบียบ", "ยึดติดของเดิม", "ไม่ชอบเปลี่ยนแปลง"],
        "background": "ข้าราชการเกษียณ อยู่กับภรรยา ประหยัดมาก ไปตลาดเช้าเป็นประจำ ภักดีต่อร้านประจำ",
        "pain_points": ["ไม่ชอบร้านไกล เดินไม่ไหว", "ไม่ชอบของแพง", "สับสนกับเทคโนโลยี"],
        "preferred_stores": ["ตลาดสด", "CJ More"],
    }
}

# แสดง agent ทั้ง 3 คน
print("Agent ทั้ง 3 คน:")
for name, data in AGENTS.items():
    p = data["personality"]
    print(f"  - {name} ({data['job']}, อายุ {data['age']}) "
          f"| ดูราคา={p['price_sensitivity']}, สะดวก={p['convenience_preference']}, โปร={p['promotion_attraction']}")

In [ ]:
# ============================================================
# ร้านค้า + โปรโมชั่น (สถานการณ์ที่ Agent จะเจอ)
# ============================================================

STORES = {
    "CJ More":   {"distance_km": 2.5, "type": "ซูเปอร์มาร์เก็ต", "promos": "นมลด 30%, ไข่ ซื้อ 2 แถม 1, ผงซักฟอกลด 25%"},
    "Big C":     {"distance_km": 4.0, "type": "ไฮเปอร์มาร์เก็ต", "promos": "นม Meiji ซื้อ 2 แถม 1"},
    "7-Eleven":  {"distance_km": 0.3, "type": "ร้านสะดวกซื้อ",   "promos": "ไม่มีโปร แต่ใกล้มาก เปิด 24 ชม."},
    "ตลาดสด":    {"distance_km": 1.0, "type": "ตลาดสด",         "promos": "ราคาถูกสุด ไม่มีโปรพิเศษ"},
    "Tops":      {"distance_km": 3.0, "type": "ซูเปอร์พรีเมียม",  "promos": "ผักออร์แกนิคลด 15%"},
}

# สถานการณ์ที่ทุกคนเจอเหมือนกัน
SITUATION = "วันเสาร์เช้า 09:00 น. นมในบ้านเหลือแค่ 1 กล่อง (ถึง threshold ต้องซื้อแล้ว)"

print("สถานการณ์:", SITUATION)
print("\nร้านค้าที่เลือกได้:")
for name, info in STORES.items():
    print(f"  - {name} ({info['distance_km']} km) | {info['type']} | {info['promos']}")

In [ ]:
# ============================================================
# Memory Classes (simplified)
# ============================================================
# ในระบบจริง Memory Stream จะเก็บประสบการณ์ทั้งหมดของ agent
# และใช้ formula: score = recency x importance x relevance
# ใน demo นี้เราใช้แบบง่ายเพื่อให้เข้าใจ concept

@dataclass
class Memory:
    """หน่วยความจำหนึ่งชิ้นของ agent"""
    content: str          # เนื้อหาความทรงจำ
    importance: int       # ความสำคัญ 1-10
    memory_type: str      # ประเภท: observation, reflection, decision

    def __repr__(self):
        return f"[{self.memory_type}] (importance={self.importance}) {self.content}"


class SimpleMemoryStream:
    """Memory Stream แบบง่าย สำหรับ demo

    ในระบบจริงจะมี:
    - Vector DB สำหรับ semantic search
    - Recency decay (ความทรงจำเก่าจะจางลง)
    - Importance scoring โดย LLM
    """
    def __init__(self):
        self.memories: List[Memory] = []

    def add(self, content: str, importance: int = 5, memory_type: str = "observation"):
        self.memories.append(Memory(content, importance, memory_type))

    def get_recent(self, n: int = 5) -> List[Memory]:
        return self.memories[-n:]

    def get_by_importance(self, min_importance: int = 7) -> List[Memory]:
        return [m for m in self.memories if m.importance >= min_importance]


# ตัวอย่างการใช้งาน
demo_memory = SimpleMemoryStream()
demo_memory.add("เห็นใบปลิว CJ More มีโปรนมลด 30%", importance=7, memory_type="observation")
demo_memory.add("เมื่อวานนมเหลือแค่ 1 กล่อง", importance=8, memory_type="observation")
demo_memory.add("ครั้งก่อนไป Big C วันเสาร์ จอดรถยาก 20 นาที", importance=6, memory_type="reflection")

print("ตัวอย่าง Memory Stream:")
for m in demo_memory.get_recent():
    print(f"  {m}")

In [ ]:
# ============================================================
# Mock Decisions สำหรับแต่ละ Agent
# ============================================================
# แต่ละคนตัดสินใจต่างกันตาม personality ของตัวเอง
# (ใช้เมื่อ USE_REAL_LLM = False)

MOCK_DECISIONS = {
    "สมศรี": {
        "thinking_steps": [
            "วันนี้วันเสาร์เช้า นมเหลือแค่กล่องเดียว ต้องซื้อเพิ่มแน่ๆ",
            "เห็นว่า CJ More มีโปรลดนม 30% น่าสนใจมาก เพราะฉันชอบดูโปรอยู่แล้ว (promotion_attraction=0.9)",
            "Big C ก็มีนม Meiji ซื้อ 2 แถม 1 แต่จำได้ว่าจอดรถยากมากวันเสาร์",
            "CJ More ยังมีโปรไข่และผงซักฟอกด้วย ซื้อได้ครบในที่เดียว ราคาคุ้ม",
            "ตัดสินใจไป CJ More เพราะโปรดีที่สุด ครบทุกอย่าง จอดรถง่ายกว่า"
        ],
        "decision": {
            "action": "go_shopping",
            "store": "CJ More",
            "items_to_buy": ["นมสด Dutch Mill (โปรลด 30%)", "ไข่ไก่ (ซื้อ 2 แถม 1)", "ผงซักฟอก (ลด 25%)"],
            "estimated_spend": 350,
            "reasoning": "เลือก CJ More เพราะมีโปรตรงกับสินค้าที่ต้องซื้อทุกอย่าง ราคาคุ้มค่า"
        }
    },
    "สมชาย": {
        "thinking_steps": [
            "นมหมดแล้ว... ขี้เกียจออกไปไกลๆ วันเสาร์อยากนอนนานหน่อย",
            "7-Eleven อยู่แค่ 300 เมตร เดินไปได้เลย ไม่ต้องขับรถ ไม่ต้องหาที่จอด",
            "ราคาแพงกว่าหน่อย (ประมาณ 75 บาท vs 45 บาทที่โปร) แต่ไม่เป็นไร เงินเดือน 80K",
            "ไม่อยากเสียเวลาขับไป CJ More 2.5 กม. เพื่อประหยัดแค่ 30 บาท (convenience=0.9)",
            "ตัดสินใจ: ไป 7-Eleven หยิบนมมา ใช้เวลาไม่ถึง 5 นาที แล้วกลับมาเขียนโค้ดต่อ"
        ],
        "decision": {
            "action": "go_shopping",
            "store": "7-Eleven",
            "items_to_buy": ["นมสด Meiji (ราคาปกติ)", "ขนมปัง", "กาแฟกระป๋อง"],
            "estimated_spend": 150,
            "reasoning": "เลือก 7-Eleven เพราะใกล้ที่สุด ไม่ต้องเสียเวลา ยอมจ่ายแพงกว่าเพื่อความสะดวก"
        }
    },
    "ลุงสมบัติ": {
        "thinking_steps": [
            "นมเหลือน้อย แต่ไม่ต้องรีบมาก ฉันกับแม่ดื่มไม่เยอะ",
            "วันเสาร์ตลาดเช้ามีนมสดขาย ราคาถูกกว่าห้างเยอะ กล่องละ 45 บาทเอง",
            "ฉันไปตลาดเช้าเป็นประจำอยู่แล้ว แม่ค้าประจำรู้จักกัน บางทีลดให้อีก (brand_loyalty=0.8)",
            "ห้างใหญ่ไม่ค่อยอยากไป ไกล คนเยอะ เดินไม่ไหว สับสนกับป้ายราคา",
            "โปร CJ More ก็น่าสนใจ แต่ต้องซื้อ 2 ชิ้น ฉันกับแม่กินไม่ทัน อาจเสียก่อน",
            "ตัดสินใจ: ไปตลาดเช้าตามปกติ ซื้อนม 1 กล่อง + ผักสด ราคาประหยัดที่สุด"
        ],
        "decision": {
            "action": "go_shopping",
            "store": "ตลาดสด",
            "items_to_buy": ["นมสดถุง (ราคาถูกสุด)", "ผักบุ้ง", "หมูชิ้น"],
            "estimated_spend": 120,
            "reasoning": "เลือกตลาดสดเพราะราคาถูกที่สุด ใกล้บ้าน คุ้นเคยกับแม่ค้า ไม่ต้องไปห้างไกลๆ"
        }
    }
}


def call_llm_for_agent(agent: Dict, situation: str) -> Dict:
    """เรียก LLM ให้ตัดสินใจสำหรับ agent นี้

    ถ้า USE_REAL_LLM = False จะใช้ mock data
    ถ้า USE_REAL_LLM = True จะเรียก OpenAI API จริง
    """
    if not USE_REAL_LLM:
        return MOCK_DECISIONS.get(agent["name"], MOCK_DECISIONS["สมศรี"])

    # ── เรียก LLM จริง ──
    try:
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            print(f"    ไม่พบ OPENAI_API_KEY กลับไปใช้ Mock")
            return MOCK_DECISIONS.get(agent["name"], MOCK_DECISIONS["สมศรี"])

        from openai import OpenAI
        client = OpenAI(api_key=api_key)

        p = agent["personality"]
        prompt = f"""คุณคือ {agent['name']} ({agent['job']}, อายุ {agent['age']} ปี)
{agent['background']}

บุคลิกภาพ (0=ต่ำ, 1=สูง):
- ดูราคา: {p['price_sensitivity']}, ภักดีแบรนด์: {p['brand_loyalty']}
- ชอบคุณภาพ: {p['quality_preference']}, เน้นสะดวก: {p['convenience_preference']}
- ชอบโปร: {p['promotion_attraction']}, ซื้อตามอารมณ์: {p['impulse_buying']}

นิสัย: {', '.join(agent.get('innate_traits', []))}
ไม่ชอบ: {', '.join(agent.get('pain_points', []))}

สถานการณ์: {situation}

ร้านค้าใกล้เคียง:
- CJ More (2.5 km) โปร: นมลด 30%, ไข่ซื้อ 2 แถม 1, ผงซักฟอกลด 25%
- Big C (4.0 km) โปร: นม Meiji ซื้อ 2 แถม 1
- 7-Eleven (0.3 km) ไม่มีโปร แต่สะดวก
- ตลาดสด (1.0 km) ราคาถูกสุด ไม่มีโปรพิเศษ
- Tops (3.0 km) ของพรีเมียม ผักออร์แกนิคลด 15%

คิดทีละขั้นตอนแล้วตัดสินใจ ตอบเป็น JSON:
{{"thinking_steps": [...], "decision": {{"action": "go_shopping", "store": "...", "items_to_buy": [...], "estimated_spend": 0, "reasoning": "..."}}}}"""

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "จำลองพฤติกรรมลูกค้าตาม personality ที่กำหนด ตอบ JSON เท่านั้น"},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            response_format={"type": "json_object"}
        )

        return json.loads(response.choices[0].message.content)

    except ImportError:
        print(f"    ไม่พบ library 'openai' กลับไปใช้ Mock")
        return MOCK_DECISIONS.get(agent["name"], MOCK_DECISIONS["สมศรี"])
    except Exception as e:
        print(f"    Error: {e} กลับไปใช้ Mock")
        return MOCK_DECISIONS.get(agent["name"], MOCK_DECISIONS["สมศรี"])


print("Mock decisions loaded for:", list(MOCK_DECISIONS.keys()))
print("LLM function ready. Mode:", "Real LLM" if USE_REAL_LLM else "Mock")

## การทดลอง: สถานการณ์เดียวกัน, Agent ต่างกัน

เราจะให้ Agent ทั้ง 3 คนเจอ **สถานการณ์เดียวกัน**:

> "วันเสาร์เช้า 09:00 น. นมในบ้านเหลือแค่ 1 กล่อง (ถึง threshold ต้องซื้อแล้ว)"

แต่ละคนจะผ่าน **Cognitive Loop** เหมือนกัน:
1. **Perceive** - รับรู้สถานการณ์ (นมหมด, ร้านค้าใกล้เคียง, โปรโมชั่น)
2. **Retrieve** - ดึงความทรงจำที่เกี่ยวข้อง
3. **Plan** - ใช้ LLM ตัดสินใจ (โดยพิจารณา personality + memory + situation)
4. **Execute** - ทำตามแผน
5. **Reflect** - สร้าง insight ใหม่

สังเกตว่าแต่ละคนจะ **คิดต่างกัน** และ **เลือกร้านต่างกัน** ทั้งที่เจอสถานการณ์เดียวกัน!

In [ ]:
# ============================================================
# Agent 1: สมศรี (แม่บ้าน) - Cognitive Loop
# ============================================================

print_header("Agent 1: สมศรี (แม่บ้าน)")

agent_data = AGENTS["สมศรี"]

# แสดง personality
print("\n[Personality Profile]")
display_personality_radar(agent_data)

# Perceive: รับรู้สถานการณ์
print(f"\n[1. Perceive] สมศรี รับรู้:")
print(f"    - นมเหลือ 1 กล่อง (ต่ำกว่า threshold)")
print(f"    - วันเสาร์เช้า (วันช้อปปิ้งประจำ)")
print(f"    - เห็นใบปลิว CJ More มีโปรลดนม 30%")

# Retrieve: ดึงความทรงจำ
print(f"\n[2. Retrieve] ความทรงจำที่เกี่ยวข้อง:")
print(f"    - ครั้งก่อนไป Big C วันเสาร์ จอดรถยาก 20 นาที (importance=6)")
print(f"    - CJ More มีโปรดีๆ บ่อย เคยประหยัดได้เยอะ (importance=7)")
print(f"    - ราคานมที่ CJ More ถูกกว่า Big C 5 บาท (importance=5)")

# Plan: ใช้ LLM ตัดสินใจ
print(f"\n[3. Plan] เรียก LLM ให้ตัดสินใจ...")
decision_somsri = call_llm_for_agent(agent_data, SITUATION)
display_thinking_process("สมศรี", decision_somsri)

# Execute & Reflect
print(f"\n[4. Execute] สมศรี ขับรถไป CJ More (2.5 km)")
print(f"[5. Reflect] \"CJ More มีโปรดี ครั้งหน้าควรดูใบปลิวก่อนไปเสมอ\"")

In [ ]:
# ============================================================
# Agent 2: สมชาย (โปรแกรมเมอร์) - Cognitive Loop
# ============================================================

print_header("Agent 2: สมชาย (โปรแกรมเมอร์)")

agent_data = AGENTS["สมชาย"]

# แสดง personality
print("\n[Personality Profile]")
display_personality_radar(agent_data)

# Perceive
print(f"\n[1. Perceive] สมชาย รับรู้:")
print(f"    - นมหมดแล้ว")
print(f"    - วันเสาร์เช้า (อยากพักผ่อน ไม่อยากออกไปไกล)")
print(f"    - 7-Eleven อยู่แค่ 300 เมตร")

# Retrieve
print(f"\n[2. Retrieve] ความทรงจำที่เกี่ยวข้อง:")
print(f"    - เคยไปห้างวันเสาร์ เสียเวลา 2 ชม. (importance=7)")
print(f"    - 7-Eleven ใช้เวลาไม่ถึง 5 นาที (importance=6)")
print(f"    - กำลังเขียนโค้ดอยู่ ไม่อยากหยุดนาน (importance=5)")

# Plan
print(f"\n[3. Plan] เรียก LLM ให้ตัดสินใจ...")
decision_somchai = call_llm_for_agent(agent_data, SITUATION)
display_thinking_process("สมชาย", decision_somchai)

# Execute & Reflect
print(f"\n[4. Execute] สมชาย เดินไป 7-Eleven (0.3 km)")
print(f"[5. Reflect] \"ซื้อที่ 7-Eleven สะดวกดี ถึงแพงหน่อยแต่ประหยัดเวลา\"")

In [ ]:
# ============================================================
# Agent 3: ลุงสมบัติ (ข้าราชการเกษียณ) - Cognitive Loop
# ============================================================

print_header("Agent 3: ลุงสมบัติ (ข้าราชการเกษียณ)")

agent_data = AGENTS["ลุงสมบัติ"]

# แสดง personality
print("\n[Personality Profile]")
display_personality_radar(agent_data)

# Perceive
print(f"\n[1. Perceive] ลุงสมบัติ รับรู้:")
print(f"    - นมเหลือน้อย (แต่อยู่ 2 คน ไม่ต้องรีบมาก)")
print(f"    - วันเสาร์เช้า (วันไปตลาดเช้าประจำ)")
print(f"    - ตลาดสดอยู่ไม่ไกล ราคาถูก")

# Retrieve
print(f"\n[2. Retrieve] ความทรงจำที่เกี่ยวข้อง:")
print(f"    - ไปตลาดเช้าทุกวันเสาร์ แม่ค้าประจำรู้จักกัน (importance=8)")
print(f"    - เคยไปห้างใหญ่ สับสนกับป้ายราคา เดินเมื่อย (importance=7)")
print(f"    - ซื้อนมถุงที่ตลาดถูกกว่าห้าง 20 บาท (importance=6)")

# Plan
print(f"\n[3. Plan] เรียก LLM ให้ตัดสินใจ...")
decision_lungsombat = call_llm_for_agent(agent_data, SITUATION)
display_thinking_process("ลุงสมบัติ", decision_lungsombat)

# Execute & Reflect
print(f"\n[4. Execute] ลุงสมบัติ เดินไปตลาดเช้า (1.0 km)")
print(f"[5. Reflect] \"ตลาดสดยังถูกสุด แม่ค้าประจำลดให้อีก ไม่ต้องไปห้างไกลๆ\"")

In [ ]:
# ============================================================
# ตารางเปรียบเทียบ Side-by-Side
# ============================================================
# รวมการตัดสินใจทั้ง 3 คน เปรียบเทียบกัน

decisions = {
    "สมศรี": decision_somsri,
    "สมชาย": decision_somchai,
    "ลุงสมบัติ": decision_lungsombat,
}

print_header("เปรียบเทียบผลลัพธ์ Side-by-Side")

# หัวตาราง
names = list(AGENTS.keys())
print(f"\n  {'─' * 64}")
header = "  "
for name in names:
    header += f"{name:^20s} "
print(header)
print(f"  {'─' * 64}")

# ข้อมูลพื้นฐาน
rows = [
    ("อาชีพ", lambda a: a["job"]),
    ("อายุ", lambda a: f"{a['age']} ปี"),
    ("รายได้", lambda a: f"{a['monthly_income']:,} บ./ด."),
    ("ครัวเรือน", lambda a: f"{a['household_size']} คน"),
]

for label, getter in rows:
    row = f"  {label:8s} "
    for name in names:
        val = getter(AGENTS[name])
        row += f"{val:^20s} "
    print(row)

print(f"  {'─' * 64}")

# Personality traits (เฉพาะที่ต่างกันชัด)
key_traits = [
    ("ดูราคา", "price_sensitivity"),
    ("เน้นสะดวก", "convenience_preference"),
    ("ชอบโปร", "promotion_attraction"),
    ("ภักดีแบรนด์", "brand_loyalty"),
    ("ซื้อตามอารมณ์", "impulse_buying"),
]

for label, trait_key in key_traits:
    row = f"  {label:8s} "
    for name in names:
        val = AGENTS[name]["personality"][trait_key]
        bar = make_bar(val, 10)
        row += f"  {bar} {val:.1f}   "
    print(row)

print(f"  {'─' * 64}")

# การตัดสินใจ
decision_rows = [
    ("ร้านที่เลือก", lambda d: d["decision"]["store"]),
    ("ใช้เงิน", lambda d: f"~{d['decision']['estimated_spend']} บาท"),
]

for label, getter in decision_rows:
    row = f"  {label:8s} "
    for name in names:
        val = getter(decisions[name])
        row += f"{val:^20s} "
    print(row)

# สินค้าที่ซื้อ
max_items = max(len(decisions[n]["decision"]["items_to_buy"]) for n in names)
for i in range(max_items):
    label = "สินค้า" if i == 0 else ""
    row = f"  {label:8s} "
    for name in names:
        items = decisions[name]["decision"]["items_to_buy"]
        if i < len(items):
            item = items[i]
            if len(item) > 18:
                item = item[:15] + "..."
            row += f"  {item:<18s} "
        else:
            row += f"  {'':18s} "
    print(row)

print(f"  {'─' * 64}")

# เหตุผล
print(f"\n  เหตุผลของแต่ละคน:")
for name in names:
    reason = decisions[name]["decision"]["reasoning"]
    print(f"    >>> {name}: {reason}")

In [ ]:
# ============================================================
# Personality Bar Comparison
# ============================================================
# เปรียบเทียบ personality trait แต่ละตัว ว่าส่งผลต่อการตัดสินใจอย่างไร

print_header("วิเคราะห์: Personality ส่งผลอย่างไร")

analyses = [
    {
        "trait": "price_sensitivity (ดูราคา)",
        "explanation": "ยิ่งสูง --> ยิ่งเลือกร้านราคาถูก / โปรดี",
        "examples": [
            ("สมศรี",     0.8,  "เลือก CJ More เพราะโปรลด 30%"),
            ("สมชาย",     0.3,  "ไม่สน ยอมซื้อแพงที่ 7-Eleven"),
            ("ลุงสมบัติ",  0.95, "เลือกตลาดสดเพราะราคาถูกที่สุด"),
        ]
    },
    {
        "trait": "convenience_preference (เน้นความสะดวก)",
        "explanation": "ยิ่งสูง --> ยิ่งเลือกร้านใกล้ / ง่าย",
        "examples": [
            ("สมศรี",     0.4,  "ยอมขับไป CJ More 2.5 กม."),
            ("สมชาย",     0.9,  "เลือก 7-Eleven ที่อยู่แค่ 300 ม."),
            ("ลุงสมบัติ",  0.3,  "ไปตลาดเช้าที่คุ้นเคย ใกล้บ้าน"),
        ]
    },
    {
        "trait": "promotion_attraction (ชอบโปร)",
        "explanation": "ยิ่งสูง --> ยิ่งสนใจร้านที่มีโปร",
        "examples": [
            ("สมศรี",     0.9,  "หาโปรเปรียบเทียบหลายร้าน"),
            ("สมชาย",     0.2,  "ไม่สนโปร ซื้อราคาปกติ"),
            ("ลุงสมบัติ",  0.6,  "สนใจโปรบ้าง แต่ไม่ถึงกับตามหา"),
        ]
    },
    {
        "trait": "brand_loyalty (ภักดีแบรนด์)",
        "explanation": "ยิ่งสูง --> ยิ่งไปร้านเดิม / ซื้อแบรนด์เดิม",
        "examples": [
            ("สมศรี",     0.6,  "เปิดใจลองร้านที่มีโปร"),
            ("สมชาย",     0.4,  "เปลี่ยนร้านได้ตามสะดวก"),
            ("ลุงสมบัติ",  0.8,  "ไปตลาดเช้าร้านเดิมเสมอ"),
        ]
    },
]

for analysis in analyses:
    print(f"\n    {analysis['trait']}")
    print(f"    {analysis['explanation']}")
    print()
    for name, score, behavior in analysis["examples"]:
        bar = make_bar(score, 10)
        print(f"      {name:8s} {bar} {score:.1f}  --> {behavior}")

## วิเคราะห์: ทำไมแต่ละคนตัดสินใจต่างกัน?

### สถานการณ์เดียวกัน แต่ผลลัพธ์ต่างกัน:

| Agent | ร้านที่เลือก | เหตุผลหลัก | Trait ที่ส่งผลมากสุด |
|-------|-------------|-----------|---------------------|
| **สมศรี** | CJ More | โปรดี ราคาคุ้ม | `promotion_attraction=0.9`, `price_sensitivity=0.8` |
| **สมชาย** | 7-Eleven | ใกล้ สะดวก ไม่เสียเวลา | `convenience_preference=0.9`, `price_sensitivity=0.3` (ไม่สนราคา) |
| **ลุงสมบัติ** | ตลาดสด | ถูกสุด ร้านประจำ | `price_sensitivity=0.95`, `brand_loyalty=0.8` |

### Key Insight:

**Personality traits ไม่ได้ทำงานแยกกัน** - LLM พิจารณาทุก trait พร้อมกัน:

- **สมศรี**: `price_sensitivity` สูง + `promotion_attraction` สูง --> เลือกร้านที่มี "โปรดี" (ไม่ใช่แค่ "ถูกสุด")
- **ลุงสมบัติ**: `price_sensitivity` สูงกว่าสมศรี + `brand_loyalty` สูง --> เลือกร้านที่ "ถูกที่สุด" + "คุ้นเคย" (ไม่สนโปร)
- **สมชาย**: `convenience_preference` สูงมาก + `price_sensitivity` ต่ำ --> ยอมจ่ายแพงเพื่อความสะดวก

นี่คือความแตกต่างระหว่าง **Rule-based** (if-else) กับ **LLM-based** (Generative Agents):
- Rule-based: `if price_sensitivity > 0.7: go_cheapest()` -- แข็ง, ไม่สมจริง
- LLM-based: พิจารณาทุกปัจจัยพร้อมกัน -- ยืดหยุ่น, สมจริง

## สรุป: สิ่งที่ได้เรียนรู้

### 1. Personality ส่งผลต่อการตัดสินใจ
- สถานการณ์เดียวกัน แต่ 3 คนเลือกร้านต่างกัน!
  - **สมศรี** --> CJ More (ราคาดี มีโปร)
  - **สมชาย** --> 7-Eleven (ใกล้ สะดวก)
  - **ลุงสมบัติ** --> ตลาดสด (ถูกสุด ร้านประจำ)

### 2. ไม่มี "คำตอบที่ถูก" - ขึ้นอยู่กับ agent
- LLM ไม่ได้ถูก hardcode ว่า "ถ้าดูราคา --> ไปตลาด"
- LLM พิจารณาหลายปัจจัยพร้อมกัน เหมือนคนจริง
- Personality เป็นแค่ "แนวโน้ม" ไม่ใช่ "กฎตายตัว"

### 3. ใน Simulation จริง:
- Agent หลายร้อยตัว แต่ละตัวมี personality ต่างกัน
- ทำให้ได้ข้อมูลที่หลากหลาย เหมือนลูกค้าจริง
- สามารถทดสอบโปรโมชั่นกับ agent กลุ่มต่างๆ ได้

### 4. ข้อดีของ LLM-based vs Rule-based:

**Rule-based (แบบเดิม):**
```python
if price_sensitivity > 0.7:
    go_to_cheapest_store()      # <-- แข็ง ไม่สมจริง
elif convenience > 0.7:
    go_to_nearest_store()
```

**LLM-based (Generative Agents):**
```
LLM พิจารณาทุกปัจจัยพร้อมกัน  # <-- ยืดหยุ่น สมจริง
+ personality + memory + situation
--> ตัดสินใจแบบ nuanced (ซับซ้อน ละเอียด)
```

---

**Generative Agents สร้าง "คนจำลอง" ที่มีเหตุผล ไม่ใช่แค่โปรแกรมที่ทำตามกฎ!**